# 03 · One-stage anchor-based detector: RetinaNet

Same protocol, code path and evaluator as Faster R-CNN (notebook 02); only the model config changes.

Experiments (incremental): **R0** torchvision defaults → **R1** anchors fitted to the training boxes → **R2** lower IoU thresholds for positive/negative anchors → **R3** focal-loss γ/α (loss modification). Optional **R4**: box loss GIoU → L1 ablation.

## 1. Setup and protocol

In [ ]:
import os, sys, json, shutil
from pathlib import Path

REPO = Path("/kaggle/working/repo") if Path("/kaggle").exists() else Path("..").resolve()
if Path("/kaggle").exists() and not REPO.exists():
    !git clone https://github.com/<user>/<repo>.git {REPO}
    !pip install -q faster-coco-eval
sys.path.insert(0, str(REPO / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as patches, cv2
import vdet, parallel, tvdet
import torch
from vdet import CLASSES

vdet.seed_everything()
ROOT = vdet.find_data_root()
OUT = Path("/kaggle/working") if Path("/kaggle").exists() else REPO / "outputs"
RUNS = OUT / "runs"; RUNS.mkdir(parents=True, exist_ok=True)
RESULT_DIRS = [RUNS, REPO / "results"]   # own runs + runs committed by the teammate
P = vdet.PROTOCOL

# restore finished runs from earlier versions of this notebook (attached as Kaggle input)
for src in (Path("/kaggle/input").glob("*/runs/*") if Path("/kaggle/input").exists() else []):
    if (src / "config.json").exists() and not (RUNS / src.name).exists():
        shutil.copytree(src, RUNS / src.name); print("restored", src.name)

In [ ]:
imgs, anns = vdet.load_all(ROOT)
folds = pd.read_csv(OUT / "folds.csv") if (OUT / "folds.csv").exists() else vdet.make_folds(imgs, P["n_folds"])
folds.to_csv(OUT / "folds.csv", index=False)   # the parallel runner reads the split from this file
train_frames, val_frames = vdet.protocol_frames(imgs, folds, P)
train_gt = anns[anns.key.isin(train_frames.key) & (anns.iscrowd == 0)]
print(json.dumps(P))
print(f"train: {len(train_frames)} frames | val: {len(val_frames)} frames / {val_frames.clip_uid.nunique()} clips")

## 2. Grounding anchors in the data

Anchors are fitted on the **training folds only** (learned preprocessing must not see the holdout). For each candidate anchor set we compute *coverage*: the share of training boxes (rescaled to `imgsz`) whose best anchor has shape-IoU ≥ 0.5 (RetinaNet's positive threshold) or ≥ 0.4. Shape-IoU ignores the position on the stride grid, so real coverage is somewhat lower, mostly for the tiniest boxes.

In [ ]:
sc = (P["imgsz"] / np.maximum(train_gt.width, train_gt.height)).values  # long side -> imgsz, as in training
wh = np.c_[train_gt.w.values * sc, train_gt.h.values * sc]

def anchor_set(base_sizes, ratios, octaves=3):
    out = []
    for s in base_sizes:
        for o in range(octaves):
            a = s * 2 ** (o / octaves)
            for r in ratios:  # torchvision: ratio = h / w
                out.append((a / np.sqrt(r), a * np.sqrt(r)))
    return np.array(out)

def coverage(wh, anchors):
    inter = np.minimum(wh[:, None, 0], anchors[None, :, 0]) * np.minimum(wh[:, None, 1], anchors[None, :, 1])
    iou = inter / (wh[:, None].prod(2) + anchors[None].prod(2) - inter)
    best = iou.max(1)
    return {"cov@0.5": (best >= .5).mean(), "cov@0.4": (best >= .4).mean(), "mean best IoU": best.mean()}

rng = np.random.default_rng(0); whs = wh[rng.choice(len(wh), min(100_000, len(wh)), replace=False)]
print("box side at imgsz: p5 / median / p95 =", np.round(np.quantile(np.sqrt(whs.prod(1)), [.05, .5, .95]), 1))
print("h/w ratio quantiles p10/p50/p90 =", np.round(np.quantile(whs[:, 1] / whs[:, 0], [.1, .5, .9]), 2))

DEFAULT = dict(base=[32, 64, 128, 256, 512], ratios=[0.5, 1.0, 2.0])
rows = [{"base": "default 32..512", "ratios": str(DEFAULT["ratios"]), **coverage(whs, anchor_set(DEFAULT["base"], DEFAULT["ratios"]))}]
for s0 in [8, 12, 16, 24]:
    for ratios in ([0.5, 1.0, 2.0], [0.5, 1.0, 2.0, 3.0], [0.33, 0.66, 1.0, 1.5, 3.0]):
        base = [s0 * 2 ** i for i in range(5)]
        rows.append({"base": f"{s0}..{base[-1]}", "s0": s0, "ratios": str(ratios), "n_ratios": len(ratios),
                     **coverage(whs, anchor_set(base, ratios))})
cov = pd.DataFrame(rows)
display(cov.style.format("{:.3f}", subset=["cov@0.5", "cov@0.4", "mean best IoU"]))

In [ ]:
# Pick the best set by coverage@0.5; prefer 3 ratios when the gain from more ratios is < 1 pt,
# because every extra ratio adds a full set of anchors (memory, more negatives).
cand = cov.dropna(subset=["s0"]).sort_values("cov@0.5", ascending=False)
best3 = cand[cand.n_ratios == 3].iloc[0]; best_any = cand.iloc[0]
chosen = best_any if best_any["cov@0.5"] - best3["cov@0.5"] >= 0.01 else best3
ANCHOR_SIZES = [int(chosen.s0) * 2 ** i for i in range(5)]
ASPECT_RATIOS = json.loads(chosen.ratios)
print("chosen anchors:", ANCHOR_SIZES, ASPECT_RATIOS, f"cov@0.5={chosen['cov@0.5']:.3f}")

## 3. Experiments

In [ ]:
BASE = dict(arch="retinanet_v2", imgsz=P["imgsz"], epochs=P["epochs"], batch=4, lr=0.01, sched="step",
            box_loss="giou", focal_gamma=2.0, focal_alpha=0.25, seed=P["seed"])
EXPERIMENTS = {"retina_r0_base": BASE}
EXPERIMENTS["retina_r1_anchors"] = {**BASE, "anchor_sizes": ANCHOR_SIZES, "aspect_ratios": ASPECT_RATIOS}
# small boxes rarely reach IoU 0.5 with any anchor -> lower thresholds give them positives
EXPERIMENTS["retina_r2_iou"] = {**EXPERIMENTS["retina_r1_anchors"], "kwargs": dict(fg_iou_thresh=0.4, bg_iou_thresh=0.3)}
FOCAL_ON = "retina_r2_iou"   # set to the better of r1/r2 once both finished
EXPERIMENTS["retina_r3_focal"] = {**EXPERIMENTS[FOCAL_ON], "focal_gamma": 1.5, "focal_alpha": 0.25}
EXPERIMENTS["retina_r4_l1"] = {**EXPERIMENTS[FOCAL_ON], "box_loss": "l1"}
pd.DataFrame(EXPERIMENTS).T[["anchor_sizes", "aspect_ratios", "kwargs", "focal_gamma", "box_loss"]]

In [ ]:
# Accelerator "GPU T4 x2": two runs train at the same time, one per GPU (quota is per session hour).
# Queue as many runs as fit into the 12 h session; finished runs are skipped. Logs: runs/<name>.log
RUN = ["retina_r0_base", "retina_r1_anchors"]
codes = parallel.run_parallel([(n, EXPERIMENTS[n]) for n in RUN], RUNS, folds_csv=OUT / "folds.csv")

## 4. Results and comparison with the tuned two-stage detector

In [ ]:
COLS = ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR_max", "ms_per_img", "epochs"]
def show(res, cols=COLS):
    cols = [c for c in cols if c in res]
    num = [c for c in cols if c not in ("ms_per_img", "epochs")]
    return res[cols].style.format("{:.4f}", subset=num).format("{:.1f}", subset=[c for c in ["ms_per_img"] if c in cols])

res = vdet.collect_results(RESULT_DIRS, "retina")
frcnn = vdet.collect_results(RESULT_DIRS, "frcnn")
if len(frcnn):
    best_frcnn = frcnn.AP.idxmax(); res = pd.concat([res, frcnn.loc[[best_frcnn]]])
if len(res): display(show(res))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for run in res.index if len(res) else []:
    h = vdet.load_history(RESULT_DIRS, run)
    if h is not None:
        axes[0].plot(h.epoch, h.AP, "o-", label=run); axes[1].plot(h.epoch, h.AP_small, "o-", label=run)
axes[0].set_title("val AP"); axes[1].set_title("val AP_small"); axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "retina_curves.png", dpi=120); plt.show()

**Does removing the proposal stage cause issues?** Fill in with the numbers above.

- *Foreground/background imbalance.* At `imgsz` = 1280 RetinaNet scores ~10⁵ anchors per image, of which only a few hundred are positive. Without an RPN filtering easy negatives, the plain cross-entropy is dominated by background. Focal loss (γ) and the prior-probability initialisation of the classification bias address it — R3 tests γ.
- *No re-pooling of features.* Faster R-CNN re-samples features for each proposal with RoIAlign and refines the box a second time; RetinaNet regresses once from a fixed anchor. For tiny objects this shows up as lower AP75 / more localisation errors — compare AP75 with the Faster R-CNN row.
- *Anchor assignment for tiny boxes.* With default 32-px anchors most boxes never reach IoU 0.5 with any anchor and get no positives at all (coverage table). Fixes: anchors from data (R1), lower IoU thresholds (R2), or adaptive assignment as in ATSS.

## 5. Export and test submission

In [ ]:
BEST = res.drop(index=[i for i in res.index if not i.startswith("retina")]).AP.idxmax() if len(res) else "retina_r0_base"
model = tvdet.load_run(RUNS / BEST)
test_preds = tvdet.predict_test(model, ROOT)
test_preds.to_csv(OUT / f"test_preds_{BEST}.csv.gz", index=False)
vdet.write_submission(test_preds, ROOT / "test_images.csv", OUT / f"submission_{BEST}.csv")

In [ ]:
# Results for the repo: unzip results.zip from the notebook Output into <repo>/results/ and commit.
vdet.export_results(RUNS, OUT / "results", preds_for=(BEST,))